# Train the EV battery detector on Colab (GPU)

Trains YOLOv8n (and optionally YOLO11n) on the **deduped 4,425-image** module/busbar
dataset assembled from ~9 independent Roboflow sources (0 cross-source duplicates,
0 test-set leakage). Runs in ~15-25 min on a Colab GPU vs ~4-5h on CPU.

**Before running:** set Runtime -> Change runtime type -> **GPU (T4/L4/A100)**.
With Colab Pro, pick an L4 or A100 for speed.

**Data:** upload `ev_detector_data.zip` (in your ~/Downloads) to your Google Drive
first (e.g. to `MyDrive/`), then run the cells below.

In [ ]:
# 1. GPU check + install
import torch, subprocess
print('CUDA available:', torch.cuda.is_available())
print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
!pip install -q ultralytics

In [ ]:
# 2. Get the data from Google Drive and unzip to /content/ev_detector_data
from google.colab import drive
drive.mount('/content/drive')

# adjust the path if you put the zip somewhere other than MyDrive/
ZIP = '/content/drive/MyDrive/ev_detector_data.zip'
!unzip -q -o "$ZIP" -d /content/
!ls /content/ev_detector_data
!echo "train images:" $(ls /content/ev_detector_data/images/train/*.jpg | wc -l)

In [ ]:
# 3. Train YOLOv8n (paper-style safe augmentation, GPU)
from ultralytics import YOLO

YAML = '/content/ev_detector_data/dataset.yaml'
model = YOLO('yolov8n.pt')
model.train(
    data=YAML, epochs=100, imgsz=640, batch=32, device=0,
    optimizer='SGD', lr0=0.01, weight_decay=0.0005,
    hsv_v=0.30, hsv_s=0.25, hsv_h=0.0,     # no hue jitter (paper: preserves condition cues)
    mosaic=0.5, degrees=2.0, translate=0.06, fliplr=0.5,
    patience=20, project='/content/runs', name='yolov8n_4425', exist_ok=True,
)

In [ ]:
# 4. Evaluate on the held-out real test split
metrics = model.val(data=YAML, split='test', imgsz=640, device=0,
                    project='/content/runs', name='yolov8n_4425_test', exist_ok=True)
print(f"TEST mAP50    = {metrics.box.map50:.3f}")
print(f"TEST mAP50-95 = {metrics.box.map:.3f}")
print("(baseline on the paper's 1,759-image data was mAP50 0.818 / mAP50-95 0.557)")

In [ ]:
# 5. (Optional) Same-budget YOLO11n comparison — one line changes
m11 = YOLO('yolo11n.pt')
m11.train(data=YAML, epochs=100, imgsz=640, batch=32, device=0,
          optimizer='SGD', lr0=0.01, weight_decay=0.0005,
          hsv_v=0.30, hsv_s=0.25, hsv_h=0.0, mosaic=0.5, degrees=2.0,
          translate=0.06, fliplr=0.5, patience=20,
          project='/content/runs', name='yolo11n_4425', exist_ok=True)
m11_metrics = m11.val(data=YAML, split='test', imgsz=640, device=0,
                      project='/content/runs', name='yolo11n_4425_test', exist_ok=True)
print(f"YOLO11n TEST mAP50 = {m11_metrics.box.map50:.3f}  mAP50-95 = {m11_metrics.box.map:.3f}")

In [ ]:
# 6. Download the trained weights (and copy to Drive for safekeeping)
from google.colab import files
best = '/content/runs/yolov8n_4425/weights/best.pt'
!cp "$best" /content/drive/MyDrive/ev_detector_yolov8n_4425_best.pt
print('Saved to Drive: MyDrive/ev_detector_yolov8n_4425_best.pt')
files.download(best)

## Back on your machine
Put the downloaded `best.pt` at
`models/detector/stage2_recall_boost/weights/best.pt` to make it the shipped
detector, then run `python scripts/evaluate.py` locally to confirm, and
`python scripts/eval_cross_variant.py` to re-check unseen-pack generalization.